# P4: State Chemical & Spending Merge

Primary merge per `MERGE.md` (§2, **P4**). Combines the two **state-grain**
agriculture tables into one wide table at **1 row per `state_fips` + `year`**.
Because both sources are Iowa-only (`state_fips == 19`), the join key collapses to
effectively **1 row per year** — there is no spatial variation to preserve.

Each source is **pivoted wide** first — its categorical descriptors
(commodity/input class/active ingredient, or expense category/unit) become
columns — so the final `state_fips` + `year` join can't fan out.

**Inputs** (both `data/tabular/02_clean/agriculture/...`):
- `crop-chemical-application-clean.csv` — pounds of active ingredient applied
  statewide, pivoted by `commodity` × `input_class` × `active_ingredient`.
- `chemical-fertilizer-feed-spending-clean.csv` — annual statewide farm
  production expenses, pivoted by `expense_category` × `unit`.

**Output:** `data/03a_merge_primary/state-chemical-spending.csv`, one row per
state + year.

**Design notes**
- **Suppressed values stay blank.** ~45% of the chemical-application rows are
  `(D)` disclosure suppressions, already blank in `value` from the cleaning step;
  the pivot carries the blanks through rather than filling `0`.
- **`active_ingredient == "TOTAL"` rows are kept** — they are the per-class
  statewide subtotals (e.g. total herbicide LB on corn) and occupy their own
  `...__total_lb` columns, distinct from the individual ingredients, so no
  double-count occurs on the pivot.
- **Spending units are remapped to readable tokens** (`$` → `usd`,
  `$ / OPERATION` → `usd_per_operation`, etc.) before they enter column names, so
  the four reporting bases for each expense category are legible.
- **Sparse by design.** Chemical application is only published for selected years
  (2015–2018, 2020, 2021, 2023); spending runs annually 2015–2024. The outer join
  unions both year sets, leaving the chemical block null in the off years —
  expected.

In [1]:
import os
import re

import pandas as pd
from functools import reduce

AG = "../../data/tabular/02_clean/agriculture"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/state-chemical-spending.csv"

KEY = ["state_fips", "year"]


def slug(x):
    """Lowercase, snake_case a categorical label for use in a column name."""
    s = str(x).strip().lower().replace("&", "and").replace("/", "_")
    s = re.sub(r"[(),.$]", "", s)
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def clean_state_year(df):
    """Standardize the join keys: 2-digit string state_fips, integer year."""
    df = df.copy()
    df["state_fips"] = pd.to_numeric(df["state_fips"], errors="coerce")
    df = df.dropna(subset=["state_fips"])
    df["state_fips"] = df["state_fips"].astype(int).astype(str).str.zfill(2)
    df["year"] = df["year"].astype(int)
    return df


def pivot_wide(df, pivot_cols, value_col, prefix, suffix=""):
    """Pivot one long table to 1 row per state_fips+year. The (state, year,
    *pivot_cols) key is asserted unique first so pivot_table never silently
    aggregates. Column names are prefix__<slug>__<slug>...<suffix>."""
    df = clean_state_year(df)
    key = KEY + pivot_cols
    dup = df.duplicated(subset=key).sum()
    assert dup == 0, f"{prefix}: {dup} duplicate rows on {key} — pivot would aggregate"
    wide = df.pivot_table(index=KEY, columns=pivot_cols, values=value_col, aggfunc="mean")
    if isinstance(wide.columns, pd.MultiIndex):
        wide.columns = [prefix + "__" + "__".join(slug(p) for p in tup) + suffix for tup in wide.columns]
    else:
        wide.columns = [prefix + "__" + slug(c) + suffix for c in wide.columns]
    wide = wide.reset_index()
    print(f"{prefix}: {wide.shape[0]:,} state-years x {wide.shape[1] - 2} value cols")
    return wide

## Step 1: Crop chemical application

Pivoted by `commodity` (CORN/SOYBEANS) × `input_class`
(HERBICIDE/FUNGICIDE/INSECTICIDE/OTHER/FERTILIZER) × `active_ingredient`. Every
value is pounds of active ingredient applied statewide (`unit == "LB"`,
`statistic == "APPLICATIONS"` — both constant, so neither is embedded in the
column name), so a single `_lb` suffix documents the unit.

In [2]:
chem = pd.read_csv(f"{AG}/crop-chemical-application-clean.csv")
assert set(chem["unit"].dropna().unique()) <= {"LB"}, "unexpected unit in chemical application"
chem_wide = pivot_wide(
    chem,
    pivot_cols=["commodity", "input_class", "active_ingredient"],
    value_col="value",
    prefix="chemapp",
    suffix="_lb",
)
chem_wide.head(3)

chemapp: 7 state-years x 98 value cols


,state_fips,year,chemapp__corn__fertilizer__nitrogen_lb,chemapp__corn__fertilizer__phosphate_lb,chemapp__corn__fertilizer__potash_lb,chemapp__corn__fertilizer__sulfur_lb,chemapp__corn__fungicide__azoxystrobin_lb,chemapp__corn__fungicide__benzovindiflupyr_lb,chemapp__corn__fungicide__metconazole_lb,chemapp__corn__fungicide__propiconazole_lb,...,chemapp__soybeans__herbicide__total_lb,chemapp__soybeans__herbicide__trifluralin_lb,chemapp__soybeans__insecticide__bifenthrin_lb,chemapp__soybeans__insecticide__chlorpyrifos_lb,chemapp__soybeans__insecticide__esfenvalerate_lb,chemapp__soybeans__insecticide__imidacloprid_lb,chemapp__soybeans__insecticide__lambda-cyhalothrin_lb,chemapp__soybeans__insecticide__total_lb,chemapp__soybeans__insecticide__zeta-cypermethrin_lb,chemapp__soybeans__other__total_lb
0,19,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,17021000.0,666000.0,44000.0,241000.0,NaN,NaN,29000.0,425000.0,NaN,NaN
1,19,2016,2.036100e+09,701100000.0,938800000.0,45300000.0,94000.0,NaN,21000.0,119000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,19887000.0,411000.0,50000.0,105000.0,7000.0,26000.0,25000.0,221000.0,NaN,NaN


## Step 2: Fertilizer / feed spending

Pivoted by `expense_category` × `unit`. The four reporting bases (`$`,
`$ / OPERATION`, `PCT OF OPERATIONS`, `PCT OF PRODUCTION EXPENSES`) are remapped
to readable tokens first so they survive as legible column suffixes rather than
`$`-laden names.

In [3]:
spend = pd.read_csv(f"{AG}/chemical-fertilizer-feed-spending-clean.csv")

UNIT_TOKEN = {
    "$": "usd",
    "$ / OPERATION": "usd_per_operation",
    "PCT OF OPERATIONS": "pct_of_operations",
    "PCT OF PRODUCTION EXPENSES": "pct_of_prod_expenses",
}
assert set(spend["unit"].unique()) <= set(UNIT_TOKEN), (
    f"unmapped spending unit: {set(spend['unit'].unique()) - set(UNIT_TOKEN)}"
)
spend = spend.copy()
spend["unit_token"] = spend["unit"].map(UNIT_TOKEN)

spend_wide = pivot_wide(
    spend,
    pivot_cols=["expense_category", "unit_token"],
    value_col="value",
    prefix="spend",
)
spend_wide.head(3)

spend: 10 state-years x 12 value cols


,state_fips,year,spend__chemical_totals__pct_of_operations,spend__chemical_totals__pct_of_prod_expenses,spend__chemical_totals__usd,spend__chemical_totals__usd_per_operation,spend__feed__pct_of_operations,spend__feed__pct_of_prod_expenses,spend__feed__usd,spend__feed__usd_per_operation,spend__fertilizer_totals_incl_lime_and_soil_conditioners__pct_of_operations,spend__fertilizer_totals_incl_lime_and_soil_conditioners__pct_of_prod_expenses,spend__fertilizer_totals_incl_lime_and_soil_conditioners__usd,spend__fertilizer_totals_incl_lime_and_soil_conditioners__usd_per_operation
0,19,2015,61.8,3.6,9.900000e+08,11314.0,39.2,18.7,5.190000e+09,59314.0,63.4,7.4,2.040000e+09,23314.0
1,19,2016,63.9,4.3,1.130000e+09,12989.0,38.0,19.8,5.210000e+09,59885.0,60.4,7.1,1.880000e+09,21609.0
2,19,2017,59.2,4.3,1.140000e+09,13240.0,40.9,16.7,4.400000e+09,51103.0,60.2,6.9,1.810000e+09,21022.0


## Step 3: Outer-merge on `state_fips` + `year`, then save

Each input is already 1 row per state-year, so an outer join on
`state_fips` + `year` unions the year coverage of the two tables without any
fan-out.

In [4]:
frames = [chem_wide, spend_wide]
df = reduce(lambda l, r: l.merge(r, on=KEY, how="outer"), frames)

assert not df.duplicated(subset=KEY).any(), "Output grain violated: duplicate (state_fips, year) rows"

df = df.sort_values(KEY).reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(f"State-years: {len(df):,}  |  distinct states: {df['state_fips'].nunique()}  |  "
      f"year range: {df['year'].min()}-{df['year'].max()}")

def block_coverage(prefix):
    cols = [c for c in df.columns if c.startswith(prefix + "__")]
    return df[cols].notna().any(axis=1).sum()

print("\nState-years with any data, by source block:")
for pfx in ["chemapp", "spend"]:
    print(f"  {pfx:8s}: {block_coverage(pfx):>3,}")

Final shape: (10, 112)
State-years: 10  |  distinct states: 1  |  year range: 2015-2024

State-years with any data, by source block:
  chemapp :   7
  spend   :  10


In [5]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 10 rows x 112 cols -> ../../data/03a_merge_primary/state-chemical-spending.csv
